In [ ]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import kagglehub

path = kagglehub.dataset_download("nelgiriyewithana/global-weather-repository")
path = path + "/GlobalWeatherRepository.csv"
df = pd.read_csv(path)
df.head(3)

100%|██████████| 10.8M/10.8M [00:00<00:00, 176MB/s]

Extracting files...


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,...,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,...,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,...,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# Output directory
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Palette & style
DARK  = "#0f1117"
PANEL = "#1a1d2e"
ACC1  = "#6c63ff"
ACC2  = "#00d4b4"
ACC3  = "#ff6b6b"
ACC4  = "#ffd166"
ACC5  = "#a8edea"
LIGHT = "#e2e8f0"
MUTED = "#718096"

plt.rcParams.update({
    "figure.facecolor": DARK,  "axes.facecolor": PANEL,
    "text.color": LIGHT,       "axes.labelcolor": LIGHT,
    "xtick.color": MUTED,      "ytick.color": MUTED,
    "axes.edgecolor": "#2d3748","grid.color": "#2d3748",
    "grid.alpha": 0.5,         "font.family": "DejaVu Sans",
    "axes.titleweight": "bold","axes.titlesize": 12,
})

# Add continent column
COUNTRY_CONTINENT = {
    # Africa
    "Algeria":"Africa","Angola":"Africa","Benin":"Africa","Botswana":"Africa",
    "Burkina Faso":"Africa","Burundi":"Africa","Cameroon":"Africa","Cape Verde":"Africa",
    "Central African Republic":"Africa","Chad":"Africa","Comoros":"Africa",
    "Congo":"Africa","Democratic Republic of Congo":"Africa","Djibouti":"Africa",
    "Egypt":"Africa","Equatorial Guinea":"Africa","Eritrea":"Africa",
    "Ethiopia":"Africa","Gabon":"Africa","Gambia":"Africa","Ghana":"Africa",
    "Guinea":"Africa","Guinea-Bissau":"Africa","Ivory Coast":"Africa",
    "Kenya":"Africa","Lesotho":"Africa","Liberia":"Africa","Libya":"Africa",
    "Madagascar":"Africa","Malawi":"Africa","Mali":"Africa","Mauritania":"Africa",
    "Mauritius":"Africa","Morocco":"Africa","Mozambique":"Africa","Namibia":"Africa",
    "Niger":"Africa","Nigeria":"Africa","Rwanda":"Africa","Sao Tome and Principe":"Africa",
    "Senegal":"Africa","Seychelles":"Africa","Sierra Leone":"Africa","Somalia":"Africa",
    "South Africa":"Africa","South Sudan":"Africa","Sudan":"Africa","Swaziland":"Africa",
    "Tanzania":"Africa","Togo":"Africa","Tunisia":"Africa","Uganda":"Africa",
    "Zambia":"Africa","Zimbabwe":"Africa",
    # Asia
    "Afghanistan":"Asia","Armenia":"Asia","Azerbaijan":"Asia","Bahrain":"Asia",
    "Bangladesh":"Asia","Bhutan":"Asia","Brunei":"Asia","Cambodia":"Asia",
    "China":"Asia","Cyprus":"Asia","Georgia":"Asia","India":"Asia",
    "Indonesia":"Asia","Iran":"Asia","Iraq":"Asia","Israel":"Asia",
    "Japan":"Asia","Jordan":"Asia","Kazakhstan":"Asia","Kuwait":"Asia",
    "Kyrgyzstan":"Asia","Laos":"Asia","Lebanon":"Asia","Malaysia":"Asia",
    "Maldives":"Asia","Mongolia":"Asia","Myanmar":"Asia","Nepal":"Asia",
    "North Korea":"Asia","Oman":"Asia","Pakistan":"Asia","Palestine":"Asia",
    "Philippines":"Asia","Qatar":"Asia","Saudi Arabia":"Asia","Singapore":"Asia",
    "South Korea":"Asia","Sri Lanka":"Asia","Syria":"Asia","Taiwan":"Asia",
    "Tajikistan":"Asia","Thailand":"Asia","Timor-Leste":"Asia","Turkey":"Asia",
    "Turkmenistan":"Asia","United Arab Emirates":"Asia","Uzbekistan":"Asia",
    "Vietnam":"Asia","Yemen":"Asia",
    # Europe
    "Albania":"Europe","Andorra":"Europe","Austria":"Europe","Belarus":"Europe",
    "Belgium":"Europe","Bosnia and Herzegovina":"Europe","Bulgaria":"Europe",
    "Croatia":"Europe","Czech Republic":"Europe","Denmark":"Europe",
    "Estonia":"Europe","Finland":"Europe","France":"Europe","Germany":"Europe",
    "Greece":"Europe","Hungary":"Europe","Iceland":"Europe","Ireland":"Europe",
    "Italy":"Europe","Kosovo":"Europe","Latvia":"Europe","Liechtenstein":"Europe",
    "Lithuania":"Europe","Luxembourg":"Europe","Malta":"Europe","Moldova":"Europe",
    "Monaco":"Europe","Montenegro":"Europe","Netherlands":"Europe",
    "North Macedonia":"Europe","Norway":"Europe","Poland":"Europe",
    "Portugal":"Europe","Romania":"Europe","Russia":"Europe","San Marino":"Europe",
    "Serbia":"Europe","Slovakia":"Europe","Slovenia":"Europe","Spain":"Europe",
    "Sweden":"Europe","Switzerland":"Europe","Ukraine":"Europe",
    "United Kingdom":"Europe","Vatican City":"Europe",
    # North America
    "Antigua and Barbuda":"North America","Bahamas":"North America",
    "Barbados":"North America","Belize":"North America","Canada":"North America",
    "Costa Rica":"North America","Cuba":"North America","Dominica":"North America",
    "Dominican Republic":"North America","El Salvador":"North America",
    "Grenada":"North America","Guatemala":"North America","Haiti":"North America",
    "Honduras":"North America","Jamaica":"North America","Mexico":"North America",
    "Nicaragua":"North America","Panama":"North America",
    "Saint Kitts and Nevis":"North America","Saint Lucia":"North America",
    "Saint Vincent and the Grenadines":"North America","Trinidad and Tobago":"North America",
    "United States":"North America","United States of America":"North America",
    # South America
    "Argentina":"South America","Bolivia":"South America","Brazil":"South America",
    "Chile":"South America","Colombia":"South America","Ecuador":"South America",
    "Guyana":"South America","Paraguay":"South America","Peru":"South America",
    "Suriname":"South America","Uruguay":"South America","Venezuela":"South America",
    # Oceania
    "Australia":"Oceania","Fiji":"Oceania","Kiribati":"Oceania",
    "Marshall Islands":"Oceania","Micronesia":"Oceania","Nauru":"Oceania",
    "New Zealand":"Oceania","Palau":"Oceania","Papua New Guinea":"Oceania",
    "Samoa":"Oceania","Solomon Islands":"Oceania","Tonga":"Oceania",
    "Tuvalu":"Oceania","Vanuatu":"Oceania",
}

df["continent"] = df["country"].map(COUNTRY_CONTINENT).fillna("Other")

# Drop rows missing critical numeric columns
key_cols = ["temperature_celsius", "humidity", "wind_kph", "pressure_mb",
            "cloud", "uv_index", "visibility_km", "precip_mm",
            "air_quality_PM2.5", "latitude", "longitude"]

df = df.dropna(subset=key_cols).reset_index(drop=True)

# Ensure air-quality columns exist (fill with NaN if absent)
aq_cols = [
    "air_quality_PM2.5", "air_quality_PM10",
    "air_quality_Carbon_Monoxide", "air_quality_Ozone",
    "air_quality_Nitrogen_dioxide", "air_quality_Sulphur_dioxide",
    "air_quality_us-epa-index",
]
for col in aq_cols:
    if col not in df.columns: df[col] = np.nan

print(f"Dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print(df[key_cols].dtypes)

# Sorted continents list
conts  = sorted(df["continent"].unique())
colors = [ACC1, ACC2, ACC3, ACC4, "#c084fc", "#fb923c", "#34d399", "#60a5fa"][:len(conts)]

# FIGURE 1 — CLIMATE ANALYSIS
fig1 = plt.figure(figsize=(20, 14), facecolor=DARK)
fig1.suptitle("Climate Analysis  —  Global Weather Repository",
              fontsize=18, color=LIGHT, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig1, hspace=0.42, wspace=0.35)

#   a. Temperature distribution per continent
ax = fig1.add_subplot(gs[0, 0])
data_t = [df[df["continent"] == c]["temperature_celsius"].dropna().values for c in conts]
bp = ax.boxplot(data_t, patch_artist=True,
                medianprops=dict(color="white", linewidth=2),
                whiskerprops=dict(color=MUTED), capprops=dict(color=MUTED),
                flierprops=dict(marker="o", markersize=2, alpha=0.3, color=MUTED))
for patch, col in zip(bp["boxes"], colors):
    patch.set_facecolor(col); patch.set_alpha(0.75)
ax.set_xticklabels([c.replace(" ", "\n") for c in conts], fontsize=7.5)
ax.set_ylabel("Temperature (°C)"); ax.set_title("Temperature Distribution by Continent")
ax.axhline(df["temperature_celsius"].mean(), color=ACC4, lw=1.5, ls="--", label="Global mean")
ax.legend(fontsize=8)

#   b. Temp vs Humidity scatter
ax = fig1.add_subplot(gs[0, 1])
sc = ax.scatter(df["humidity"], df["temperature_celsius"],
                c=df["uv_index"], cmap="plasma", alpha=0.35, s=6, linewidths=0)
cb = plt.colorbar(sc, ax=ax); cb.set_label("UV Index", color=LIGHT)
cb.ax.yaxis.set_tick_params(color=LIGHT)
m, b, r, p, _ = stats.linregress(df["humidity"], df["temperature_celsius"])
xs = np.linspace(df["humidity"].min(), df["humidity"].max(), 100)
ax.plot(xs, m*xs+b, color=ACC2, lw=2, label=f"R² = {r**2:.3f}")
ax.set_xlabel("Humidity (%)"); ax.set_ylabel("Temperature (°C)")
ax.set_title("Temperature vs Humidity (UV Index coloured)")
ax.legend(fontsize=9)

#   c. Rainfall by continent
ax = fig1.add_subplot(gs[0, 2])
rain_cont = df.groupby("continent")["precip_mm"].mean().sort_values(ascending=True)
bars = ax.barh(rain_cont.index, rain_cont.values,
               color=colors[:len(rain_cont)], alpha=0.85)
for bar, val in zip(bars, rain_cont.values):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
            f"{val:.1f} mm", va="center", fontsize=8, color=LIGHT)
ax.set_xlabel("Avg Precipitation (mm)"); ax.set_title("Average Precipitation by Continent")

#   d. Wind-Pressure bivariate
ax = fig1.add_subplot(gs[1, 0])
sc2 = ax.scatter(df["pressure_mb"], df["wind_kph"],
                 c=df["cloud"], cmap="cool", alpha=0.3, s=6)
cb2 = plt.colorbar(sc2, ax=ax); cb2.set_label("Cloud Cover (%)", color=LIGHT)
ax.set_xlabel("Pressure (mb)"); ax.set_ylabel("Wind Speed (kph)")
ax.set_title("Wind Speed vs Atmospheric Pressure")

#   e. UV Index distribution
ax = fig1.add_subplot(gs[1, 1])
ax.hist(df["uv_index"], bins=30, color=ACC4, alpha=0.8, edgecolor="none")
ax.axvline(df["uv_index"].mean(), color=ACC3, lw=2,
           label=f'Mean {df["uv_index"].mean():.1f}')
ax.set_xlabel("UV Index"); ax.set_ylabel("Count")
ax.set_title("UV Index Distribution"); ax.legend(fontsize=9)

#   f. Visibility by continent
ax = fig1.add_subplot(gs[1, 2])
vis_df = df.groupby("continent")["visibility_km"].agg(["mean", "std"]).reset_index()
vis_df.sort_values("mean", inplace=True)
ax.barh(vis_df["continent"], vis_df["mean"], xerr=vis_df["std"],
        color=ACC5, alpha=0.8, error_kw=dict(ecolor=MUTED, capsize=3))
ax.set_xlabel("Visibility (km)"); ax.set_title("Avg Visibility by Continent (±1 SD)")

plt.savefig(f"{OUTPUT_DIR}/01_climate_analysis.png",
            dpi=160, bbox_inches="tight", facecolor=DARK)
plt.close()
print("✓ Figure 1 saved")

# FIGURE 2 — ENVIRONMENTAL IMPACT (AIR QUALITY)
fig2 = plt.figure(figsize=(20, 14), facecolor=DARK)
fig2.suptitle("Environmental Impact  —  Air Quality & Weather Correlations",
              fontsize=18, color=LIGHT, fontweight="bold", y=0.98)
gs2 = gridspec.GridSpec(2, 3, figure=fig2, hspace=0.42, wspace=0.38)

aq_plot_cols = ["air_quality_PM2.5", "air_quality_PM10",
                "air_quality_Carbon_Monoxide", "air_quality_Ozone",
                "air_quality_Nitrogen_dioxide", "air_quality_Sulphur_dioxide"]
wx_cols = ["temperature_celsius", "humidity", "wind_kph",
           "pressure_mb", "cloud", "visibility_km", "uv_index", "precip_mm"]

#   a. Correlation heatmap (AQ vs weather)
ax = fig2.add_subplot(gs2[0, :2])
corr_data = df[aq_plot_cols + wx_cols].corr().loc[aq_plot_cols, wx_cols]
short_aq = ["PM2.5", "PM10", "CO", "Ozone", "NO₂", "SO₂"]
short_wx = ["Temp", "Humidity", "Wind", "Pressure", "Cloud", "Visibility", "UV", "Precip"]
cmap = LinearSegmentedColormap.from_list("rg", [ACC3, "#2d3748", ACC2])
sns.heatmap(corr_data, ax=ax, cmap=cmap, center=0,
            annot=True, fmt=".2f", annot_kws={"size": 9},
            xticklabels=short_wx, yticklabels=short_aq,
            linewidths=0.5, linecolor="#0f1117",
            cbar_kws={"shrink": 0.7, "label": "Pearson r"})
ax.set_title("Air Quality Pollutants vs Weather Parameters — Correlation Matrix")

#   b. PM2.5 by country (top 10)
ax = fig2.add_subplot(gs2[0, 2])
pm_country = df.groupby("country")["air_quality_PM2.5"].mean().nlargest(10)
gradient = plt.cm.Reds(np.linspace(0.4, 0.9, len(pm_country)))
bars = ax.barh(pm_country.index[::-1], pm_country.values[::-1], color=gradient[::-1])
for bar, val in zip(bars, pm_country.values[::-1]):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f"{val:.0f}", va="center", fontsize=8, color=LIGHT)
ax.set_xlabel("Avg PM2.5 (μg/m³)"); ax.set_title("Top-10 Countries by PM2.5")

#   c. EPA index pie
ax = fig2.add_subplot(gs2[1, 0])
epa_labels = {1: "Good", 2: "Moderate", 3: "Unhealthy\n(Sensitive)",
              4: "Unhealthy", 5: "Very\nUnhealthy", 6: "Hazardous"}
epa_counts = df["air_quality_us-epa-index"].dropna().astype(int).value_counts().sort_index()
epa_colors = [ACC2, "#a8edea", ACC4, ACC3, "#c084fc", "#ef4444"]
wedges, texts, auts = ax.pie(
    epa_counts.values,
    labels=[epa_labels.get(i, "") for i in epa_counts.index],
    colors=epa_colors[:len(epa_counts)], autopct="%1.1f%%",
    pctdistance=0.75, textprops={"fontsize": 7, "color": LIGHT},
    wedgeprops={"linewidth": 0.5, "edgecolor": DARK})
ax.set_title("US EPA Air Quality Index Distribution")

#   d. PM2.5 vs Temperature scatter
ax = fig2.add_subplot(gs2[1, 1])
sc3 = ax.scatter(df["temperature_celsius"], df["air_quality_PM2.5"],
                 c=df["humidity"], cmap="YlOrRd", alpha=0.35, s=7)
cb3 = plt.colorbar(sc3, ax=ax); cb3.set_label("Humidity %", color=LIGHT)
m, b, r, p, _ = stats.linregress(df["temperature_celsius"],
                                 df["air_quality_PM2.5"])
xs = np.linspace(df["temperature_celsius"].min(), df["temperature_celsius"].max(), 100)
ax.plot(xs, m*xs+b, color=ACC1, lw=2, label=f"R²={r**2:.3f}")
ax.set_xlabel("Temperature (°C)"); ax.set_ylabel("PM2.5 (μg/m³)")
ax.set_title("PM2.5 vs Temperature"); ax.legend(fontsize=9)

#   e. Wind vs PM2.5
ax = fig2.add_subplot(gs2[1, 2])
wind_bins = pd.cut(df["wind_kph"], bins=6)
pm_wind = df.groupby(wind_bins, observed=True)["air_quality_PM2.5"].mean()
xs_w = range(len(pm_wind))
ax.plot(xs_w, pm_wind.values, color=ACC2, marker="o", lw=2, ms=8)
ax.fill_between(xs_w, pm_wind.values, alpha=0.2, color=ACC2)
ax.set_xticks(xs_w)
ax.set_xticklabels([str(i)[:10] for i in pm_wind.index],
                   rotation=30, ha="right", fontsize=7)
ax.set_ylabel("Avg PM2.5 (μg/m³)")
ax.set_title("PM2.5 vs Wind Speed Bins\n(higher wind → dispersion)")

plt.savefig(f"{OUTPUT_DIR}/02_environmental_impact.png",
            dpi=160, bbox_inches="tight", facecolor=DARK)
plt.close()
print("✓ Figure 2 saved")

# FIGURE 3 — FEATURE IMPORTANCE
feat_cols = ["humidity", "wind_kph", "pressure_mb", "cloud",
             "uv_index", "visibility_km", "precip_mm",
             "air_quality_PM2.5", "air_quality_Ozone",
             "air_quality_Carbon_Monoxide", "latitude"]
target = "temperature_celsius"

Xf = df[feat_cols].fillna(df[feat_cols].median())
yf = df[target]

# Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=8, n_jobs=-1, random_state=42)
rf.fit(Xf, yf)
rf_imp = pd.Series(rf.feature_importances_, index=feat_cols).sort_values(ascending=True)

# Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=150, max_depth=4, random_state=42)
gb.fit(Xf, yf)
gb_imp = pd.Series(gb.feature_importances_, index=feat_cols).sort_values(ascending=True)

# Permutation importance
perm = permutation_importance(rf, Xf, yf, n_repeats=15, random_state=42, n_jobs=-1)
perm_imp = pd.Series(perm.importances_mean, index=feat_cols).sort_values(ascending=True)

# PCA variance
scaler = StandardScaler()
Xsc = scaler.fit_transform(Xf)
pca = PCA()
pca.fit(Xsc)
var_ratio = pca.explained_variance_ratio_

# Correlation with target
corr_target = df[feat_cols].corrwith(df[target]).abs().sort_values(ascending=True)

fig3 = plt.figure(figsize=(20, 14), facecolor=DARK)
fig3.suptitle("Feature Importance  —  Predicting Temperature",
              fontsize=18, color=LIGHT, fontweight="bold", y=0.98)
gs3 = gridspec.GridSpec(2, 3, figure=fig3, hspace=0.42, wspace=0.38)

#   a. RF importance
ax = fig3.add_subplot(gs3[0, 0])
ax.barh(rf_imp.index, rf_imp.values, color=ACC1, alpha=0.85)
ax.set_title("Random Forest — Feature Importance")
ax.set_xlabel("Importance Score")

#   b. GB importance
ax = fig3.add_subplot(gs3[0, 1])
ax.barh(gb_imp.index, gb_imp.values, color=ACC2, alpha=0.85)
ax.set_title("Gradient Boosting — Feature Importance")
ax.set_xlabel("Importance Score")

#   c. Permutation importance
ax = fig3.add_subplot(gs3[0, 2])
ax.barh(perm_imp.index, perm_imp.values,
        xerr=perm.importances_std[np.argsort(perm.importances_mean)],
        color=ACC3, alpha=0.85, error_kw=dict(ecolor=MUTED, capsize=3))
ax.set_title("Permutation Importance (±SD)")
ax.set_xlabel("Mean Accuracy Decrease")

#   d. PCA scree
ax = fig3.add_subplot(gs3[1, 0])
cumvar = np.cumsum(var_ratio)
ax.bar(range(1, len(var_ratio)+1), var_ratio*100, color=ACC4, alpha=0.8)
ax2_ = ax.twinx()
ax2_.plot(range(1, len(var_ratio)+1), cumvar*100,
          color=ACC2, marker="o", ms=5, lw=2)
ax2_.set_ylabel("Cumulative Variance %", color=ACC2)
ax2_.axhline(95, color=ACC2, lw=1, ls="--", alpha=0.5)
ax.set_xlabel("Principal Component"); ax.set_ylabel("Explained Variance %")
ax.set_title("PCA Scree Plot")

#   e. Correlation with target
ax = fig3.add_subplot(gs3[1, 1])
ax.barh(corr_target.index, corr_target.values, color=ACC5, alpha=0.85)
ax.set_title("Absolute Pearson Correlation with Temperature")
ax.set_xlabel("|r|")

#   f. Ranked comparison heatmap
ax = fig3.add_subplot(gs3[1, 2])
rank_df = pd.DataFrame({
    "RF Rank":   rf_imp.rank(ascending=False),
    "GB Rank":   gb_imp.rank(ascending=False),
    "Perm Rank": perm_imp.rank(ascending=False),
    "Corr Rank": corr_target.rank(ascending=False),
})
sns.heatmap(rank_df.sort_values("RF Rank"), ax=ax,
            cmap="RdYlGn_r", annot=True, fmt=".0f",
            linewidths=0.5, linecolor=DARK,
            cbar_kws={"label": "Rank (1=most important)"})
ax.set_title("Feature Rank Comparison\nAcross Techniques")

plt.savefig(f"{OUTPUT_DIR}/03_feature_importance.png",
            dpi=160, bbox_inches="tight", facecolor=DARK)
plt.close()
print("✓ Figure 3 saved")

# FIGURE 4 — SPATIAL ANALYSIS
fig4 = plt.figure(figsize=(20, 15), facecolor=DARK)
fig4.suptitle("Spatial Analysis  —  Geographical Weather Patterns",
              fontsize=18, color=LIGHT, fontweight="bold", y=0.98)
gs4 = gridspec.GridSpec(2, 2, figure=fig4, hspace=0.38, wspace=0.30)

#   a. Temperature world scatter map
ax = fig4.add_subplot(gs4[0, :])
sc4 = ax.scatter(df["longitude"], df["latitude"],
                 c=df["temperature_celsius"], cmap="RdYlBu_r",
                 alpha=0.5, s=12, linewidths=0)
cb4 = plt.colorbar(sc4, ax=ax, fraction=0.025, pad=0.01)
cb4.set_label("Temperature (°C)", color=LIGHT)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Global Temperature Map")
ax.axhline(0,     color=MUTED, lw=0.7, ls="--")
ax.axhline( 23.5, color=ACC4, lw=0.7, ls=":", alpha=0.6, label="Tropics")
ax.axhline(-23.5, color=ACC4, lw=0.7, ls=":", alpha=0.6)
ax.legend(fontsize=8, loc="lower left")
ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)

#   b. PM2.5 spatial bubble
ax = fig4.add_subplot(gs4[1, 0])
sc5 = ax.scatter(df["longitude"], df["latitude"],
                 c=df["air_quality_PM2.5"], cmap="hot",
                 alpha=0.45, s=df["air_quality_PM2.5"]/15,
                 linewidths=0)
cb5 = plt.colorbar(sc5, ax=ax, fraction=0.04)
cb5.set_label("PM2.5 (μg/m³)", color=LIGHT)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("PM2.5 Pollution — Spatial Distribution")
ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)

#   c. Latitude vs Temperature with regression bands
ax = fig4.add_subplot(gs4[1, 1])
lat_abs = df["latitude"].abs()
sc6 = ax.scatter(lat_abs, df["temperature_celsius"],
                 c=df["humidity"], cmap="Blues", alpha=0.3, s=8)
cb6 = plt.colorbar(sc6, ax=ax); cb6.set_label("Humidity %", color=LIGHT)
m, b, r, p, se = stats.linregress(lat_abs, df["temperature_celsius"])
xs6 = np.linspace(0, 80, 100)
ax.plot(xs6, m*xs6+b, color=ACC1, lw=2.5, label=f"R²={r**2:.3f}")

ax.fill_between(xs6,
                (m*xs6+b) - 2*se*np.sqrt(len(df)),
                (m*xs6+b) + 2*se*np.sqrt(len(df)),
                alpha=0.15, color=ACC1)
ax.set_xlabel("Absolute Latitude (°)"); ax.set_ylabel("Temperature (°C)")
ax.set_title("Temperature vs Latitude — Latitudinal Climate Gradient")
ax.legend(fontsize=9)

plt.savefig(f"{OUTPUT_DIR}/04_spatial_analysis.png",
            dpi=160, bbox_inches="tight", facecolor=DARK)
plt.close()
print("✓ Figure 4 saved")

# FIGURE 5 — GEOGRAPHICAL PATTERNS
fig5 = plt.figure(figsize=(22, 16), facecolor=DARK)
fig5.suptitle("Geographical Patterns  —  Country & Continent Comparisons",
              fontsize=18, color=LIGHT, fontweight="bold", y=0.98)
gs5 = gridspec.GridSpec(2, 3, figure=fig5, hspace=0.45, wspace=0.38)

# a. Multi-metric country radar (select 6 representative countries)
metrics  = ["temperature_celsius", "humidity", "wind_kph", "uv_index", "air_quality_PM2.5", "precip_mm"]
m_labels = ["Temp", "Humidity", "Wind", "UV", "PM2.5", "Precip"]

top6     = df.groupby("country")[metrics].mean()
norm6    = (top6 - top6.min()) / (top6.max() - top6.min())

# Pick representative countries that are present in the dataset
default_sel = ["India", "United States", "Brazil", "Russia", "Australia", "Egypt"]
sel_countries = [c for c in default_sel if c in norm6.index]
# Pad with any available countries if some are missing
if len(sel_countries) < 6:
    extras = [c for c in norm6.index if c not in sel_countries]
    sel_countries += extras[:6 - len(sel_countries)]

angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]

ax_r = fig5.add_subplot(gs5[0, 0], polar=True)
ax_r.set_facecolor(PANEL)
rad_colors = [ACC1, ACC2, ACC3, ACC4, "#c084fc", "#fb923c"]
for cc, col in zip(sel_countries, rad_colors):
    vals = norm6.loc[cc, metrics].tolist() + [norm6.loc[cc, metrics[0]]]
    ax_r.plot(angles, vals, color=col, lw=2, label=cc)
    ax_r.fill(angles, vals, alpha=0.07, color=col)
ax_r.set_xticks(angles[:-1]); ax_r.set_xticklabels(m_labels, size=8)
ax_r.set_title("Country Profile Radar\n(normalised metrics)", pad=18)
ax_r.legend(loc="lower right", fontsize=7, bbox_to_anchor=(1.35, -0.05))
ax_r.tick_params(colors=MUTED)

#    b. Continent-level grouped bar
ax = fig5.add_subplot(gs5[0, 1:])
cont_mean = df.groupby("continent")[["temperature_celsius", "humidity",
                                     "wind_kph", "air_quality_PM2.5"]].mean()
x = np.arange(len(cont_mean))
w = 0.2
cols5b   = [ACC1, ACC2, ACC3, ACC4]
labels5b = ["Temp (°C)", "Humidity (%)", "Wind (kph)", "PM2.5 (μg/m³)"]
for i, (col_, lab_) in enumerate(zip(cols5b, labels5b)):
    ax.bar(x + i*w, cont_mean.iloc[:, i], width=w, label=lab_,
           color=col_, alpha=0.85)
ax.set_xticks(x + 1.5*w)
ax.set_xticklabels(cont_mean.index, rotation=20, ha="right")
ax.set_title("Mean Climate Metrics by Continent")
ax.legend(fontsize=8, ncol=2)

#   c. Temperature violin per continent
ax = fig5.add_subplot(gs5[1, 0])
vp = ax.violinplot(
    [df[df["continent"] == c]["temperature_celsius"].dropna().values for c in conts],
    positions=range(len(conts)), showmedians=True)
for body, col in zip(vp["bodies"], colors):
    body.set_facecolor(col); body.set_alpha(0.65)
vp["cmedians"].set_color("white")
vp["cmins"].set_color(MUTED); vp["cmaxes"].set_color(MUTED); vp["cbars"].set_color(MUTED)
ax.set_xticks(range(len(conts)))
ax.set_xticklabels([c.replace(" ", "\n") for c in conts], fontsize=7.5)
ax.set_ylabel("Temperature (°C)"); ax.set_title("Temperature Violin by Continent")

#   d. Country Z-score heatmap
ax = fig5.add_subplot(gs5[1, 1])
heat_cols = ["temperature_celsius", "humidity", "wind_kph",
             "uv_index", "air_quality_PM2.5", "visibility_km"]
country_pivot = df.groupby("country")[heat_cols].mean()
country_pivot_norm = (country_pivot - country_pivot.mean()) / country_pivot.std()
sns.heatmap(country_pivot_norm, ax=ax,
            cmap=LinearSegmentedColormap.from_list("bwr2", [ACC3, "#1a1d2e", ACC1]),
            center=0, linewidths=0.3, linecolor=DARK,
            yticklabels=True,
            xticklabels=["Temp", "Hum", "Wind", "UV", "PM2.5", "Vis"],
            cbar_kws={"label": "Z-score", "shrink": 0.8})
ax.set_title("Country-Level Z-Score Heatmap")
ax.set_yticklabels(ax.get_yticklabels(), fontsize=7)

#   e. Continental PM2.5 box
ax = fig5.add_subplot(gs5[1, 2])
data_pm = [df[df["continent"] == c]["air_quality_PM2.5"].dropna().values for c in conts]
bp2 = ax.boxplot(data_pm, patch_artist=True,
                 medianprops=dict(color="white", lw=2),
                 whiskerprops=dict(color=MUTED), capprops=dict(color=MUTED),
                 flierprops=dict(marker=".", markersize=2, alpha=0.3, color=MUTED))
for patch, col in zip(bp2["boxes"], colors):
    patch.set_facecolor(col); patch.set_alpha(0.75)
ax.set_xticklabels([c.replace(" ", "\n") for c in conts], fontsize=7.5)
ax.set_ylabel("PM2.5 (μg/m³)"); ax.set_title("PM2.5 Distribution by Continent")
ax.axhline(35, color=ACC3, lw=1.5, ls="--", label="WHO guideline (35)")
ax.legend(fontsize=8)

plt.savefig(f"{OUTPUT_DIR}/05_geographical_patterns.png",
            dpi=160, bbox_inches="tight", facecolor=DARK)
plt.close()
print("✓ Figure 5 saved")

# FIGURE 6 — SUMMARY DASHBOARD
fig6 = plt.figure(figsize=(20, 11), facecolor=DARK)
fig6.suptitle("Executive Summary Dashboard  —  Global Weather Repository",
              fontsize=18, color=LIGHT, fontweight="bold", y=0.98)
gs6 = gridspec.GridSpec(2, 4, figure=fig6, hspace=0.45, wspace=0.38)

def kpi_box(ax, title, value, unit, color):
    ax.set_facecolor(PANEL)
    ax.axis("off")
    ax.text(0.5, 0.72, title,  ha="center", va="center", transform=ax.transAxes,
            fontsize=10, color=MUTED, fontweight="bold")
    ax.text(0.5, 0.42, value,  ha="center", va="center", transform=ax.transAxes,
            fontsize=28, color=color, fontweight="bold")
    ax.text(0.5, 0.18, unit,   ha="center", va="center", transform=ax.transAxes,
            fontsize=9,  color=MUTED)
    rect = FancyBboxPatch((0.04, 0.04), 0.92, 0.92, boxstyle="round,pad=0.02",
                          linewidth=2, edgecolor=color, facecolor="none",
                          transform=ax.transAxes, clip_on=False)
    ax.add_patch(rect)

kpis = [
    ("Avg Temperature",   f'{df["temperature_celsius"].mean():.1f}',  "°C",    ACC1),
    ("Avg Humidity",      f'{df["humidity"].mean():.1f}',             "%",     ACC2),
    ("Avg Wind Speed",    f'{df["wind_kph"].mean():.1f}',             "kph",   ACC3),
    ("Avg PM2.5",         f'{df["air_quality_PM2.5"].mean():.1f}',    "μg/m³", ACC4),
    ("Avg UV Index",      f'{df["uv_index"].mean():.1f}',             "index", ACC5),
    ("Avg Precipitation", f'{df["precip_mm"].mean():.1f}',            "mm",    "#c084fc"),
    ("Avg Visibility",    f'{df["visibility_km"].mean():.1f}',        "km",    "#fb923c"),
    ("Countries Covered", f'{df["country"].nunique()}',                "nations","#34d399"),
]
positions = [(0,0),(0,1),(0,2),(0,3),(1,0),(1,1),(1,2),(1,3)]
for (r, c), (title, val, unit, col) in zip(positions, kpis):
    ax_ = fig6.add_subplot(gs6[r, c])
    kpi_box(ax_, title, val, unit, col)

plt.savefig(f"{OUTPUT_DIR}/06_summary_dashboard.png",
            dpi=160, bbox_inches="tight", facecolor=DARK)
plt.close()
print("✓ Figure 6 saved")
print(f"\nAll outputs written to ./{OUTPUT_DIR}/")

Using Colab cache for faster access to the 'global-weather-repository' dataset.
Dataset: 136828 rows × 42 columns
temperature_celsius    float64
humidity                 int64
wind_kph               float64
pressure_mb            float64
cloud                    int64
uv_index               float64
visibility_km          float64
precip_mm              float64
air_quality_PM2.5      float64
latitude               float64
longitude              float64
dtype: object
✓ Figure 1 saved
✓ Figure 2 saved
✓ Figure 3 saved
✓ Figure 4 saved
✓ Figure 5 saved
✓ Figure 6 saved

🎉  All outputs written to ./outputs/
